In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo_name = "flipkart-wired-x-campus-node"
cwd = Path.cwd()
if (cwd / ".git").exists() and cwd.name == repo_name:
    repo = cwd
else:
    base = Path("/content") if Path("/content").exists() else cwd
    repo = base / repo_name
    if (repo / ".git").exists():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/mba25015-maker/flipkart-wired-x-campus-node.git", str(repo)],
            check=True,
        )

os.chdir(repo)
if os.environ.get("WIRED_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(f"Repository ready: {repo}")

## What this notebook proves

This notebook applies the campus-node screening logic to public-safe AISHE district aggregates and adds the contestedness classification.

It reproduces the 72,352-institution universe, district coverage, 111-district screen, and the disqualification pressure from incumbent density.

Expected headline result: **111 candidate districts.**

In [ ]:
from pathlib import Path
from html import escape
from IPython.display import HTML, display
import subprocess, sys

model_dir = str(Path("Model").resolve())
if model_dir not in sys.path:
    sys.path.insert(0, model_dir)

import aishe_district as district_model

print(f"Institutions: {district_model.N_HEI:,} | District pairs: {district_model.N_DISTRICTS:,} | Candidate districts: {district_model.N_CANDIDATES:,}")
print(f"Urban colleges: {district_model.URBAN_COL:,} | Rural colleges: {district_model.RURAL_COL:,} | Urban share: {district_model.URBAN_SHARE_COL:.1%}")
print("Contestedness: " + " | ".join(f"{name} {count}" for name, count in district_model.PROX_COUNTS.items()))

print(f"\n{'DISTRICT':<24} {'STATE':<18} {'URBAN COLLEGES':>15} {'GEOMETRY':>16}")
print("-" * 78)
for _, row in district_model.TOP.head(10).iterrows():
    print(f"{row.District[:23]:<24} {row.State[:17]:<18} {row.urban_colleges:>15.0f} {row.geometry:>16}")

full_result = subprocess.run([sys.executable, "Model/aishe_district.py"], text=True, capture_output=True)
full_text = full_result.stdout + (("\nSTDERR\n" + full_result.stderr) if full_result.stderr.strip() else "")
display(HTML(
    "<details style='margin-top:12px'><summary><b>Show full district-screen report</b></summary>"
    f"<pre style='white-space:pre-wrap'>{escape(full_text)}</pre></details>"
))
full_result.check_returncode()

## How to read the result

The first table shows district ranking; the contestedness section separates uncontested, contested, and stacked candidates.

District enrolment is not observed in the register. Residential intensity is a state-level ratio applied to districts, and incumbent stores are an imputed scale band rather than geocoded proximity.